In [ ]:
!pip install transformers

In [ ]:
!pip install "transformers[torch]"

In [6]:
import pandas as pd
from transformers import T5Tokenizer , Trainer , TrainingArguments , T5ForConditionalGeneration

In [8]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

# Data Preprocessing

In [9]:
train_data = train_data.sample(n = 4000 , random_state = 42).reset_index(drop = True)
val_data = val_data.sample(n = 500 , random_state = 42).reset_index(drop = True)

In [10]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # Removing extra lines
    text = re.sub(r"\s+", " " , text) # Removing extra spaces
    text = re.sub(r"<.*?>"," ", text) # Removing html tags
    text = text.strip().lower()
    return text

In [11]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenizer

In [12]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [13]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"] , padding = "max_length" , max_length = 512 , truncation = True)
    targets = tokenizer(data["summary"] , padding = "max_length" , max_length = 150 , truncation = True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [14]:
train_dataset = train_data.apply( tokenize , axis = 1).tolist()
val_dataset = val_data.apply( tokenize , axis = 1).tolist()

# Model (t5-small)

In [15]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1525.87it/s]


# Device setup

In [16]:
import torch

In [17]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [18]:
print(device)

cpu


# Training Arguments

In [19]:
training_args = TrainingArguments(
    output_dir="./results",                  # Folder where model checkpoints and training results are saved

    num_train_epochs=6,                      # Number of times the model will go through the entire training dataset

    weight_decay=0.01,                       # Regularization to reduce overfitting by penalizing large model weights

    per_device_train_batch_size=8,           # Number of training samples processed at once per device (GPU/CPU)

    per_device_eval_batch_size=8,            # Number of evaluation samples processed at once per device

    eval_strategy="epoch",                   # Evaluate the model after every training epoch

    save_strategy="epoch",                   # Save a model checkpoint after every training epoch

    warmup_steps=500                         # Number of initial training steps where the learning rate gradually increases
)

# Trainer

In [20]:
trainer = Trainer(
    model=model,                    # Model to train
    args=training_args,             # Training settings
    train_dataset=train_dataset,    # Training data
    eval_dataset=val_dataset,       # Validation data
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.647280,0.381734


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,3.647280,0.381734
2,0.396649,0.359618
3,0.373707,0.354343
4,0.361280,0.350503
5,0.355205,0.349402
6,0.351106,0.348865


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9142043813069661, metrics={'train_runtime': 1280.6205, 'train_samples_per_second': 18.741, 'train_steps_per_second': 2.343, 'total_flos': 3248203235328000.0, 'train_loss': 0.9142043813069661, 'epoch': 6.0})

# Saving trained model

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

# Loading model and tokenizer

In [21]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1627.77it/s]


# Summary Function 

In [22]:
def summarize_dialogue(dialogue):
  # Pre-Processing
  dialogue = clean_data(dialogue)

  # Tokenize
  inputs = tokenizer(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"
  ).to(device)

  # Generate Summary => (token ids)
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 150,
      num_beams = 4,
      early_stopping = True
  )

  # Decoding targets
  summary = tokenizer.decode(targets[0] , skip_special_tokens = True)
  return summary

# Testing 

In [23]:
test_dialogue = """ Interviewer: Welcome to today's technology discussion. Today we're talking about the growing impact of artificial intelligence on modern businesses.

Expert: Artificial intelligence is becoming an important part of many industries. Companies are using AI to automate repetitive tasks, analyze large amounts of data, and improve customer experiences.

Interviewer: Which industries are seeing the biggest benefits from AI?

Expert: Healthcare, finance, retail, manufacturing, and transportation are among the major industries adopting AI. In healthcare, for example, AI can assist doctors in analyzing medical images and identifying potential diseases.

Interviewer: What are some of the challenges associated with AI?

Expert: One major concern is data privacy. AI systems often require large amounts of data, so companies need to make sure that personal and sensitive information is handled responsibly.

Interviewer: Are there other concerns besides privacy?

Expert: Yes. Bias is another important issue. If an AI model is trained on biased or unbalanced data, its predictions may also become biased. There are also concerns about job displacement as automation becomes more capable.

Interviewer: How can organizations use AI responsibly?

Expert: Organizations should carefully evaluate their training data, test models for bias, protect user information, and make AI decisions as transparent as possible. Human oversight is also important, especially when AI is used in high-impact situations.

Interviewer: What do you expect from AI in the future?

Expert: AI will likely become more integrated into everyday business operations. However, its success will depend not only on technological improvements but also on responsible development, appropriate regulation, and collaboration between governments, researchers, and companies. """

In [24]:
summary = summarize_dialogue(test_dialogue)
print("Summary : ",summary)

Summary :  experts are talking about the growing impact of artificial intelligence on modern businesses. healthcare, finance, retail, manufacturing, and transportation are among the major industries adopting ai.
